In [10]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.preprocessing import normalize

ratings = pd.read_csv("../data/ml-latest-small/ratings.csv")
filmes = pd.read_csv("../data/movies_enriched.csv")
titulos = filmes.set_index("movieId")["title"]

In [11]:
ratings = ratings.sample(frac=1, random_state=42).reset_index(drop=True)
ratings["pos"] = ratings.groupby("userId").cumcount()
ratings["n_user"] = ratings.groupby("userId")["rating"].transform("count")

teste_mask = ratings["pos"] < ratings["n_user"] * 0.2
teste = ratings[teste_mask].copy()
treino = ratings[~teste_mask].copy()

relevantes = (
    teste[teste["rating"] >= 4.0]
    .groupby("userId")["movieId"]
    .apply(set)
    .to_dict()
)

print("Treino:", len(treino), "| Teste:", len(teste))
print("Usuários avaliáveis:", len(relevantes))

Treino: 80419 | Teste: 20417
Usuários avaliáveis: 603


In [12]:
user_ids = np.sort(ratings["userId"].unique())
movie_ids = np.sort(ratings["movieId"].unique())
u_idx = pd.Series(np.arange(len(user_ids)), index=user_ids)
m_idx = pd.Series(np.arange(len(movie_ids)), index=movie_ids)


def montar_matriz(df, coluna):
    return csr_matrix(
        (df[coluna].values, (u_idx[df["userId"]].values, m_idx[df["movieId"]].values)),
        shape=(len(user_ids), len(movie_ids)),
    )


treino["um"] = 1
treino["rating_c"] = treino["rating"] - treino.groupby("userId")["rating"].transform("mean")

R_bin = montar_matriz(treino, "um")
R_like = montar_matriz(treino[treino["rating"] >= 4], "um")
R_c = montar_matriz(treino, "rating_c")

vistos = R_bin.toarray() > 0
n_treino = np.asarray(R_bin.sum(axis=0)).ravel()

print("Matriz usuários x filmes:", R_bin.shape)

Matriz usuários x filmes: (610, 9724)


In [13]:
scores_pop = np.tile(n_treino, (len(user_ids), 1)).astype(float)

Xn = normalize(R_c.T.tocsr())
scores_cf = ((R_like @ Xn) @ Xn.T).toarray()

print(scores_pop.shape, scores_cf.shape)

(610, 9724) (610, 9724)


In [14]:
K = 10
MIN_AVAL = 20


def avaliar(scores, K=K, min_aval=MIN_AVAL):
    s = scores.astype(float).copy()
    s[vistos] = -np.inf
    s[:, n_treino < min_aval] = -np.inf

    top = np.argpartition(-s, K, axis=1)[:, :K]
    ordem = np.argsort(-np.take_along_axis(s, top, axis=1), axis=1)
    top = np.take_along_axis(top, ordem, axis=1)

    precisoes, recalls, recomendados = [], [], set()
    for uid, rel in relevantes.items():
        recs = set(movie_ids[top[u_idx[uid]]])
        acertos = len(recs & rel)
        precisoes.append(acertos / K)
        recalls.append(acertos / len(rel))
        recomendados |= recs

    return {
        "Precision@10": np.mean(precisoes),
        "Recall@10": np.mean(recalls),
        "Cobertura": len(recomendados) / len(movie_ids),
    }

In [15]:
resultados = pd.DataFrame({
    "Popularidade (baseline)": avaliar(scores_pop),
    "Colaborativo item-item": avaliar(scores_cf),
}).T.round(4)

resultados

,Precision@10,Recall@10,Cobertura
Popularidade (baseline),0.1227,0.1079,0.0058
Colaborativo item-item,0.1028,0.0995,0.0646


In [16]:
for m in [5, 20, 50]:
    r = avaliar(scores_cf, min_aval=m)
    print(f"min_aval={m}:", {k: round(v, 4) for k, v in r.items()})

min_aval=5: {'Precision@10': np.float64(0.0796), 'Recall@10': np.float64(0.0843), 'Cobertura': 0.0988}
min_aval=20: {'Precision@10': np.float64(0.1028), 'Recall@10': np.float64(0.0995), 'Cobertura': 0.0646}
min_aval=50: {'Precision@10': np.float64(0.1345), 'Recall@10': np.float64(0.1137), 'Cobertura': 0.0307}
